# Notes as per Opencode

Just take them and give it back to opencode for formatting or any AI

Deep Dive: The Mental Model Before LangChain
1. LLM as a unit
Tokens — the real currency
LLMs don't read characters or words; they read tokens. A tokenizer (like OpenAI's tiktoken or Llama's BPE) splits text into subword units.
"LangChain is powerful!"
→ ["Lang", "Chain", " is", " power", "ful", "!"]
Rules of thumb:
- ~1 token ≈ 4 characters of English ≈ 0.75 words.
- "1000 words" ≈ 1333 tokens.
- Non-English / code / emoji cost more tokens (sometimes 2–3×).
Why you care: every token is billed and counted against the context window. A "cheap" mistake of sending a 50k-token doc every call burns money and hits limits.
You can count tokens locally:
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o")
tokens = enc.encode("LangChain is powerful!")
len(tokens)  # → number of token ids
Vocabulary
The model has a fixed vocab (e.g. GPT-4o ~128k tokens, Llama-3 ~128k). Anything outside gets fragmented into subwords. Rare words → many tokens → more cost, sometimes worse understanding.
Context window — the hard ceiling
The model processes a single flat sequence of tokens: [system][user][assistant][user]... all concatenated. The window (e.g. 8k/32k/128k/200k) is the max length of that whole sequence.
|<- context window (max N tokens) ->|
[system | user | assistant | user | ← model can only see this far
- Older turns get truncated (dropped from the front) when you exceed it.
- This is the #1 reason RAG chunks documents and retrieves only the relevant slices — to fit inside the window.
- "Long context" (200k) ≠ "infinite memory." Attention degrades on very long inputs ("lost in the middle" problem).
Pre-training vs Inference — the division of labor
Phase	Who does it	What happens
Pre-training	Lab (OpenAI/Google/Meta)	Scrape trillions of tokens → predict next token → learn weights
Fine-tuning / RLHF	Lab	Align to instructions, safety
Inference	You	Load prompt → model samples tokens → response
You never train. You only feed prompts and read completions. Your ML training experience is irrelevant to using LLMs — but useful for understanding them.
Completion vs Chat models
- Completion (gpt-3.5-turbo-instruct): single string in, text out. Legacy. "The capital of France is" → " Paris".
- Chat (gpt-4o, ChatOpenAI): a list of message objects, each with a role. This is the modern interface and what LangChain's chat models wrap.
# Raw OpenAI SDK (what LangChain hides)
from openai import OpenAI
client = OpenAI()
client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "What is RAG?"},
    ]
)
LangChain turns that dict-list into a typed ChatPromptTemplate + ChatModel object — same call, more composable.
2. Generation ≠ prediction
Your ML paradigm (discriminative)
f(x) = ŷ. Input features → a single output (class label or scalar). Deterministic: same x → same ŷ. One forward pass.
LLM paradigm (generative, autoregressive)
The model is a conditional probability distribution over the next token given everything so far:
P(token_t | token_1, token_2, ..., token_{t-1})
Generation is a loop:
1. Feed context → model outputs a distribution over the vocab.
2. Sample one token from that distribution.
3. Append it to context. Repeat until stop token or max length.
So the output is a sampled sequence, not a single prediction. This is why LLMs can "hallucinate" — they're always predicting the plausible next token, not retrieving ground truth.
The sampling knobs (this is your control panel)
- temperature scales the logits before softmax:
- 0 → greedy (always pick highest prob; deterministic, repetitive).
- 0.7 → balanced creative.
- ≥1.5 → high entropy, chaotic, often nonsense.
- top_p (nucleus sampling): only consider the smallest set of tokens whose cumulative prob ≥ p. Cuts off the long tail of unlikely tokens.
- top_k: only consider top-k tokens. (Less common now.)
- max_tokens: hard cap on output length.
- seed (some APIs): fix it + temp 0 for reproducibility.
# temp 0 = locked answer; temp 1 = varies
{"temperature": 0, "top_p": 1}   # deterministic, best for extraction/code
{"temperature": 0.8, "top_p": 0.9} # creative, best for brainstorming
Key takeaway: same prompt + temp>0 = different answers. Build apps assuming nondeterminism (parse outputs, validate, retry).
3. API consumption
The shift: local training → remote service
In ML you ran model.fit(X, y) on your machine. With LLMs you send an HTTP request to a provider's server and get tokens back. You are a client.
API key — your identity & billing
- Issued by the provider (OpenAI, Anthropic, or local Ollama which needs none).
- Format: sk-... (OpenAI). Treat like a password.
- If leaked → someone runs up your bill.
.env + python-dotenv (NEVER hardcode keys)
# .env  (gitignored!)
OPENAI_API_KEY=sk-abc123

# main.py
from dotenv import load_dotenv
import os
load_dotenv()
key = os.getenv("OPENAI_API_KEY")
Add .env to .gitignore. Use a secrets manager in production.
Rate limits
Providers cap you: e.g. 10k requests/min, 1M tokens/min. Burst past it → 429 Too Many Requests. SDKs/LangChain retry with backoff. For batches, throttle or use batch endpoints.
Cost = tokens (both directions)
You pay for input tokens (your prompt) + output tokens (the reply). Prices per 1M tokens:
- GPT-4o input ~$2.50, output ~$10 (example tiers).
- A 30k-token RAG prompt × 1000 users/day = real money. Cache embeddings, shrink prompts, use smaller models for easy tasks.
Local alternative: Ollama
Your course covers Ollama — runs open models (Llama 3, Mistral) locally, no API key, no cost, but needs RAM/GPU and is slower. LangChain swaps ChatOpenAI ↔ ChatOllama with one line — that's the "abstraction" payoff.
4. Prompt = interface
Your "code" is now natural language. The model has no hidden state between calls — everything it knows for this turn is in the messages.
Message roles (the chat protocol)
Role	Purpose	Example
system	Sets persona, rules, output format. Highest authority.	"You are a JSON-only API. Never explain."
user	The actual task/question.	"Summarize the contract in 3 bullets."
assistant	Prior model replies — used to continue a conversation.	(from a previous turn)
Few-shot prompting
Show examples to teach the format instead of describing it:
user: "en -> fr: cat"  assistant: "chat"
user: "en -> fr: dog"  assistant: "chien"
user: "en -> fr: book" assistant: ???   # model continues the pattern
PromptTemplate (LangChain's core abstraction)
Variables filled at runtime:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role} expert."),
    ("user", "Explain {topic} in {n} sentences."),
])
prompt.invoke({"role": "history", "topic": "RAG", "n": 2})
Output is the fully-rendered message list, ready for the model.
Principle: clear structure + explicit format + examples beats vague instructions. This is the programming layer of GenAI.
5. Why a framework (LangChain)
Raw SDK call works, so why LangChain? Because real apps need orchestration:
1. Provider abstraction — write once, swap models:
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
llm = ChatOpenAI(model="gpt-4o")      # or ChatOllama(model="llama3")
2. Chaining (LCEL) — compose with the | pipe (your next section):
chain = prompt | llm | StrOutputParser()
chain.invoke({"topic": "RAG"})
3. Memory — inject prior turns so the bot "remembers" (your "agents with convo history" section).
4. Tools / Agents — let the LLM decide to call functions (search, DB, calculator) — the "agentic AI" section.
5. Retries, fallbacks, streaming, tracing — production concerns handled for you.
Critical reframe: LangChain is plumbing. The LLM is the brain; LangChain routes data between the LLM, your documents, tools, and users. If you confuse the two, you'll debug the wrong layer.
6. Embeddings intuition (for RAG)
What an embedding is
A separate embedding model (not the chat model) maps text → a fixed-length vector of floats:
"dog"        → [0.21, -0.44, 0.88, ...]  (1536 dims for OpenAI)
"puppy"      → [0.20, -0.41, 0.85, ...]  ← close to "dog"
"stock mkt"  → [-0.77, 0.12, -0.30, ...] ← far from "dog"
Semantically similar text → nearby vectors. This is your stats/linear-algebra intuition applied to language.
Similarity = cosine
import numpy as np
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
# 1.0 = identical meaning, 0 = unrelated, -1 = opposite
The RAG pipeline (preview)
DOCUMENTS
  → load (PDF, web, DB)
  → split into chunks (~500 tokens)
  → embed each chunk → vector
  → store in vector DB (FAISS/Chroma/pgvector)

QUERY
  → embed query → vector
  → vector DB: similarity_search → top-k nearest chunks
  → stuff chunks into prompt
  → LLM generates answer grounded in retrieved text
This defeats hallucination + context-window limits: you give the model just the relevant docs. You already have the cosine-math; RAG is just applying it to text at scale.
Pre-LangChain checklist (verify before video 1)
- Token: can explain it, count tokens, and state that context windows cap input.
- Message roles: can write correct system / user / assistant messages.
- Env setup: can pip install, create a venv, and load a key from .env via python-dotenv.
- Reframe: can state "LangChain = plumbing around an LLM, not the LLM."
- Sampling: knows temperature/top_p change output deterministically vs creatively.
- Embeddings: understands text→vector and cosine = similarity (prep for RAG).

Points 1–3: The LLM Unit, Generation, and API
1. LLM as a unit — with extra focus on vocab & context
Tokens (quick recap)
The model reads tokens, not words or letters. A tokenizer chops text into subword pieces:
"LangChain is powerful!" → ["Lang", "Chain", " is", " power", "ful", "!"]
~4 chars ≈ 1 token ≈ 0.75 words. Everything (cost, limits) is counted in tokens.
Vocabulary — let's slow down here
Think of the vocabulary as the model's fixed dictionary of building blocks. Imagine a language where you're only allowed to use 100,000 pre-approved "pieces" — and every word you say must be built by snapping those pieces together.
- Each "piece" is a token, and the full set of allowed pieces is the vocab (e.g. GPT-4o ≈ 128,000 tokens).
- Common words are one piece: "cat" = 1 token.
- Uncommon/long words get split: "unbelievable" → ["un", "believ", "able"] = 3 tokens.
- The model never sees raw letters — it only works with token IDs (numbers). "cat" might be token ID 2571.
Why this matters to you:
- Unknown = fragmented. If a word isn't in the vocab, it's broken into subwords. More pieces = more tokens = more cost, sometimes worse understanding.
- You can't add words. Unlike your ML features (you choose columns), the vocab is frozen by the lab. You can't "teach" it a new term directly — you explain the term in the prompt instead.
- Your stats/ML intuition helps: the model learned, during pre-training, a giant vector for each token ID. Tokens that appear in similar contexts get similar vectors. That's the seed of "meaning."
Analogy: vocab = the Lego set you're given. You can only build with those bricks. A rare brick you don't have must be approximated by combining smaller bricks.
Context window — slow down here too
The model is amnesiac between calls. It has no memory. For each single call, it can only "see" one continuous strip of tokens — the context window. That strip contains everything: the system instruction + your question + any past conversation + any documents you stuffed in.
┌─────────────── CONTEXT WINDOW (max e.g. 32,000 tokens) ───────────────┐
│ [system] [user] [assistant] [user] [document chunk] [user question]  │
└──────────────────────────────────────────────────────────────────────┘
                     ↑ the model only reads THIS, left to right
Key points:
- Fixed size. If your input + desired output exceeds it, the model can't process it. Extra text from the front gets dropped (truncated).
- Bills both ways & limits you. Big window (128k) ≠ infinite. Attention also gets weaker on long inputs ("lost in the middle" — models forget stuff in the center).
- This is the root cause of RAG. You can't paste your whole company wiki (too big). So you retrieve only the relevant chunks and fit them in the window.
Analogy: the context window is the model's desk. It can only work with what's physically on the desk right now. The desk has a max size. RAG is like having a librarian run and fetch only the 3 relevant books instead of wheeling in the whole library (which won't fit).
So: vocab = the bricks it can use; context window = the desk it can see at once.
Pre-training vs inference (recap table)
Phase	Who	You?
Pre-training (learn weights)	Lab	No
Fine-tuning/RLHF (align)	Lab	No
Inference (prompt→tokens)	You	Yes
Completion vs Chat
- Completion: one string in → text out (legacy).
- Chat: a list of role-tagged messages (system/user/assistant) in → text out. This is what LangChain's ChatModel wraps.
client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "What is RAG?"},
    ]
)
2. Generation ≠ prediction
Your ML: f(x) = ŷ — one input, one deterministic output (label/number). Same x → same answer, always.
LLM: it's a next-token probability machine. Given the tokens so far, it outputs a probability for every token in the vocab, then samples one:
P(next_token | everything so far)
The loop:
1. See context → get probability distribution over all vocab tokens.
2. Sample one token (using temperature/top_p).
3. Append it, repeat until stop or max length.
So output = a sampled sequence, not a single prediction. The model always predicts the plausible next token — which is exactly why it can hallucinate (plausible ≠ true).
The knobs
Knob	Effect
temperature 0	greedy, locked, deterministic (code/extraction)
temperature ~0.8	creative, varied
top_p	nucleus sampling — ignore the unlikely tail
max_tokens	cap output length
seed (some)	reproducibility with temp 0
{"temperature": 0,   "top_p": 1}   # deterministic
{"temperature": 0.8, "top_p": 0.9} # creative
Takeaway: same prompt + temp>0 → different answers. Design apps for nondeterminism (validate/parse/retry).
3. API consumption
The shift
ML: model.fit(X,y) runs locally on your machine. LLMs: you send an HTTP request to a provider's server and get tokens back. You are a client, not a trainer.
API key
- Your identity + billing. Format sk-.... Secret — never commit to git.
- Leaked key = someone spends your money.
.env (never hardcode)
# .env  (add to .gitignore!)
OPENAI_API_KEY=sk-abc123

# main.py
from dotenv import load_dotenv
import os
load_dotenv()
key = os.getenv("OPENAI_API_KEY")
Rate limits
Caps like 10k req/min. Exceed → 429. SDKs/LangChain retry with backoff. Throttle batches.
Cost = tokens, both directions
You pay for input tokens (prompt) + output tokens (reply). A 30k-token RAG prompt × many users = real money. Cache embeddings, shrink prompts, use small models for easy tasks.
Ollama (local, your course covers it)
Runs open models (Llama 3, Mistral) locally — no key, no cost, but needs RAM/GPU and is slower. LangChain swaps ChatOpenAI ↔ ChatOllama with one line.
That's 1–3. Want me to send 4–6 (Prompt interface, Why framework, Embeddings + checklist) as the next message?

Pre-training vs Inference — what even is that?
You actually already know this split from ML. In ML you have:
- Training = learn patterns from data (model.fit).
- Inference = use the trained model to predict (model.predict).
LLMs use the same two-phase idea, just at a different scale and with different mechanics. Let me rebuild it from your ML knowledge.
Pre-training (the "training" phase — done by the lab, NOT you)
What it actually is
The lab takes a transformer model (a big neural net with random weights) and feeds it a mountain of text — basically the entire public internet, books, code. The task is deceptively simple:
Given the first part of a sentence, predict the next token.
Input:  "The capital of France is"
Target: " Paris"
Input:  "The capital of France is P"
Target: "aris"  (it learns token by token)
That's it. No labels, no human annotating "this is a geography question." It's self-supervised: the text itself provides the answer. By doing this trillions of times, the model's weights get tuned so that it "knows" facts, grammar, logic, code — all encoded implicitly in the numbers of its weight matrix.
How this differs from your ML training
Your ML training	LLM pre-training
Dataset: thousands–millions of rows	Dataset: trillions of tokens
Task: classify / regress a label	Task: predict next token
Features hand-picked by you	Raw text, learns its own representations
Runs on your laptop/colab	Runs on thousands of GPUs for weeks, costs millions
You do this	A big lab does this (OpenAI, Meta, Google)
So the critical point: pre-training is not something you participate in. It already happened before you ever wrote a line of LangChain. When you use gpt-4o, you're using a model that was pre-trained at enormous cost by someone else.
After pre-training: the model "knows" stuff
After pre-training, the model has absorbed patterns of language and a compressed version of world knowledge. But it's not yet good at following your instructions — it's just a "next-token guesser." So labs add:
- Supervised fine-tuning (SFT): show it examples of instruction → good answer so it learns the chat format.
- RLHF (Reinforcement Learning from Human Feedback): humans rank answers; model nudged toward helpful/safe ones.
Together these turn the raw next-token model into the assistant you talk to. Still all done by the lab.
Inference (the "predict" phase — THIS is what you do)
What it is
Inference = using the already-trained model to generate output. In your ML terms, it's model.predict(x). In LLM terms, it's: feed a prompt → model samples tokens → you get text back.
# YOU do this part:
response = llm.invoke("Explain RAG like I'm 5")
That llm.invoke(...) is an inference call. No learning happens. The weights are frozen. You're just running the trained network forward to get an output.
The generation loop (inference mechanics)
Unlike your ML model which outputs one value, inference for an LLM is a loop:
1. You send: [system][user]  → these are tokens
2. Model computes probabilities over its vocab for token #1
3. Samples one token (using temperature/top_p)
4. Appends it: [system][user][assistant: "R"]
5. Recomputes probabilities for token #2, given everything so far
6. Samples "A", then "G", ... until stop token
7. Returns the full text "RAG is like..."
Each step is one forward pass of the network. The model never "remembers" previous calls — every call is stateless; all context must be re-sent in the prompt (that's why memory/conversation history has to be explicitly managed, and why context window matters).
Why "inference" trips people up
- In ML, inference is fast and cheap (one matrix multiply on a small model).
- In LLMs, inference is expensive compute (generating 1000 tokens = 1000 forward passes on a huge model). That's why you pay per token and why local Ollama is slower than API.
- You don't control the weights, only the input and the sampling settings (temp/top_p). Your "levers" are the prompt + knobs, not the model parameters.
The clean mental split
┌───────────────────── DONE BY THE LAB (before you exist in the story) ─────────────────────┐
│ Pre-training (next-token on internet) → SFT (chat format) → RLHF (helpful/safe)          │
│ Result: a frozen weights file = "gpt-4o" / "llama3"                                      │
└──────────────────────────────────────────────────────────────────────────────────────────┘
                                         ↓ you download / API-call it
┌───────────────────── DONE BY YOU (this is your whole job in GenAI) ──────────────────────┐
│ Inference: craft prompt + set temp/top_p + (later) chain docs/tools via LangChain        │
│ Result: generated text answering your user's need                                        │
└──────────────────────────────────────────────────────────────────────────────────────────┘
The one-sentence version: Pre-training is the lab spending millions to make the brain; inference is you talking to that brain through a prompt. You will never pre-train in this course — you only do inference, orchestrated by LangChain.

Points 4–6: Prompt Interface, Why Framework, Embeddings
4. Prompt = interface
Your "code" in GenAI is natural language. The model has no memory between calls, so everything it needs to know for this turn must be in the messages. The prompt is your programming layer.
Message roles (the chat protocol)
Role	Job
system	Persona, rules, output format. Highest authority. Set once.
user	The actual task/question.
assistant	Previous model replies — replayed to continue a chat.
The order matters: system first, then the conversation. The model reads this whole list left-to-right as its context.
Few-shot prompting
Instead of describing the format, show it:
user: en→fr: cat     assistant: chat
user: en→fr: dog     assistant: chien
user: en→fr: book    assistant: ???   → model continues the pattern
Examples often beat instructions. This is your ML instinct (give training examples) applied to prompting.
PromptTemplate — LangChain's core abstraction
Variables filled at runtime so prompts are reusable:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role} expert."),
    ("user", "Explain {topic} in {n} sentences."),
])
prompt.invoke({"role": "history", "topic": "RAG", "n": 2})
# → rendered message list, ready for the model
Principle: explicit structure + format + examples > vague instructions.
5. Why a framework (LangChain)
A raw SDK call works in 5 lines. Why add LangChain? Because real apps need orchestration:
1. Provider abstraction — write once, swap models:
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
llm = ChatOpenAI(model="gpt-4o")   # or ChatOllama(model="llama3")
2. Chaining (LCEL) — compose with | (your next section):
chain = prompt | llm | StrOutputParser()
chain.invoke({"topic": "RAG"})
3. Memory — inject prior turns so the bot "remembers" (your conversation-history section).
4. Tools / Agents — let the LLM call functions (search, DB) — the "agentic AI" section.
5. Retries, fallbacks, streaming, tracing — production concerns handled for you.
Critical reframe: LangChain is plumbing. The LLM is the brain; LangChain routes data between the LLM, your documents, and tools. Confuse the two and you'll debug the wrong layer.
6. Embeddings intuition (for RAG)
What an embedding is
A separate embedding model maps text → a fixed-length vector of floats:
"dog"   → [0.21, -0.44, 0.88, ...]   (1536 dims for OpenAI)
"puppy" → [0.20, -0.41, 0.85, ...]   ← close to "dog"
"market"→ [-0.77, 0.12, -0.30, ...]  ← far from "dog"
Similar meaning → nearby vectors. This is your linear-algebra intuition applied to language.
Similarity = cosine
import numpy as np
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
# 1.0 = same meaning, 0 = unrelated, -1 = opposite
The RAG pipeline (preview for later)
DOCUMENTS → load → split into chunks (~500 tokens)
          → embed each chunk → store in vector DB (FAISS/Chroma)

QUERY → embed → vector DB similarity_search → top-k chunks
      → stuff chunks into prompt → LLM answers grounded in retrieved text
This beats hallucination + context limits: you give the model only the relevant docs. You already have the cosine math; RAG just applies it to text at scale.
Pre-LangChain checklist (verify before video 1)
- Token: can explain it, count tokens, and state context windows cap input.
- Message roles: can write correct system / user / assistant messages.
- Env: can pip install, make a venv, load a key from .env via python-dotenv.
- Reframe: "LangChain = plumbing around an LLM, not the LLM."
- Sampling: knows temperature/top_p change deterministic vs creative output.
- Embeddings: understands text→vector, cosine = similarity (prep for RAG).

Few-shot, Temperature/Top_p, and Vectors/Similarity — explained deeper
1. Few-shot prompting (your example, step by step)
First, the term. "Shot" = an example you show the model. Zero-shot = no example (just the task). One-shot = one example. Few-shot = 2–3 examples. The model is great at pattern matching from context, so showing examples teaches the format better than describing it.
Your example, rebuilt:
user:     "en→fr: cat"
assistant: "chat"

user:     "en→fr: dog"
assistant: "chien"

user:     "en→fr: book"
assistant: ???
What's happening:
1. You're establishing a pattern: input is "en→fr: <english word>", output is <french word>.
2. The model reads turns 1 and 2 and infers the rule (translate English to French).
3. On turn 3, it continues the pattern and answers "livre".
You never said "translate." You just demonstrated twice, and the model generalized. That's few-shot.
Why it beats instructions sometimes:
- Telling it "output only JSON" can be ignored. Showing two JSON examples makes it copy the shape reliably.
- Great for rigid formats (extract fields, classify labels, convert schemas).
# LangChain few-shot via messages
prompt = ChatPromptTemplate.from_messages([
    ("user", "en→fr: cat"),
    ("assistant", "chat"),
    ("user", "en→fr: dog"),
    ("assistant", "chien"),
    ("user", "en→fr: {word}"),   # filled at runtime
])
The assistant turns are fake prior replies you inject to set the pattern. The model treats them as "this is how this conversation goes."
Rule of thumb: if output format matters, show 2–3 examples before the real input.
2. Temperature & top_p (the sampling knobs, deeper)
Recall: after seeing the context, the model outputs a probability for every token in its vocab. Sampling = picking one token from that distribution. The knobs reshape the distribution before picking.
Temperature
Think of the raw scores (logits) for each token. Temperature divides them before turning them into probabilities:
- temp → 0: divides by ~0 → the highest-scoring token becomes ~100% probable → always picked (greedy). Deterministic, repetitive, safe for code/extraction.
- temp = 1: unchanged distribution (the model's honest beliefs).
- temp → 2: divides by a big number → flattens the distribution → all tokens more equally likely → random, creative, sometimes nonsense.
Intuition: temp controls how much the model sticks to its top guess vs explores.
top_p (nucleus sampling)
Instead of temperature's reshape, top_p cuts the vocab:
- Sort tokens by probability, take the smallest set whose cumulative probability ≥ p.
- Example: p = 0.9 → keep only the top tokens that make up 90% of the mass; discard the long tail of weird tokens.
- Then sample only from that kept set.
So temp = how sharp the distribution is; top_p = how many candidate tokens are even allowed.
How they interact
- Common creative setting: temperature=0.8, top_p=0.9 → varied but not insane.
- Common exact setting: temperature=0, top_p=1 → locked, reproducible (good for tests/parsing).
- You usually tune one or the other, not both aggressively.
temp 0, top_p 1   → always "Paris"
temp 0.8, top_p 0.9 → "Paris" / "the French capital" / "France's capital"
temp 2, top_p 1   → "baguette metropolis" (garbage)
3. Vectors & similarity (embeddings, deeper)
What a vector is here
An embedding model converts text into a list of numbers, e.g. 1536 of them:
"dog" → [0.21, -0.44, 0.88, 0.03, ...]   (1536 numbers)
Each number is a coordinate in a 1536-dimensional space. A word = a point in that space. Your ML background: this is just a feature vector, but learned so that meaning maps to geometry.
Why similar meaning = nearby points
During training, the embedding model was optimized so texts used in similar contexts end up close together in this space:
- "dog" and "puppy" → very close points.
- "dog" and "stock market" → far apart points.
You don't interpret individual dimensions ("what does coordinate 42 mean?"). You only care about distance between points = similarity of meaning.
Cosine similarity (the actual metric)
We measure closeness by the angle between two vectors, not raw distance:
import numpy as np
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
- 1.0 → identical direction (same meaning)
- 0.0 → unrelated (90° apart)
- -1.0 → opposite meaning
Cosine is used (not Euclidean distance) because it's length-invariant — a long detailed sentence and a short one about the same topic still score high.
How RAG uses this (concrete)
1. Chunk doc → "The cat sat on the mat." → embed → vector V1
2. Store V1 in a vector DB alongside thousands of other chunk-vectors.
3. User asks: "Where did the cat sit?" → embed → vector Q
4. DB computes cosine(Q, V) for every stored vector.
5. Top match: V1 (cosine ≈ 0.91) → retrieve that chunk.
6. Stuff chunk into prompt → LLM answers "on the mat."
No keyword matching ("cat" must appear) — it's meaning matching. That's the power of embeddings, and it's exactly your cosine/vector-space math from ML, applied to language.